# 02 — Перегляд датасету construction-ppe-small
Візуальна перевірка YOLO-розмітки (bounding boxes + класи).

In [ ]:
from pathlib import Path
import random, yaml, cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches

In [ ]:
dataset_path = Path("../datasets/construction-ppe-small")
with open(dataset_path / "data.yaml", "r", encoding="utf-8") as f:
    data = yaml.safe_load(f)
class_names = data["names"]
if isinstance(class_names, dict):
    class_names = {int(k): v for k, v in class_names.items()}
else:
    class_names = {i: name for i, name in enumerate(class_names)}
class_names

In [ ]:
def show_random_image(split="train"):
    images_dir = dataset_path / "images" / split
    labels_dir = dataset_path / "labels" / split
    image_paths = list(images_dir.glob("*.jpg")) + list(images_dir.glob("*.png"))
    image_path = random.choice(image_paths)
    label_path = labels_dir / f"{image_path.stem}.txt"
    image = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)
    ih, iw = image.shape[:2]
    fig, ax = plt.subplots(figsize=(12, 8)); ax.imshow(image); ax.axis("off")
    ax.set_title(f"{split}: {image_path.name}")
    if label_path.exists():
        for line in label_path.read_text().splitlines():
            p = line.split()
            if len(p) != 5: continue
            c = int(float(p[0])); xc, yc, bw, bh = map(float, p[1:])
            x = (xc - bw/2)*iw; y = (yc - bh/2)*ih
            ax.add_patch(patches.Rectangle((x, y), bw*iw, bh*ih, linewidth=2, edgecolor="red", facecolor="none"))
            ax.text(x, y-5, class_names.get(c, f"class_{c}"), fontsize=10, color="white",
                    bbox=dict(facecolor="red", alpha=0.8, pad=2))
    plt.show()

show_random_image("train")
show_random_image("val")